# Traditional ML Experiments - Experiment 2: Time Features

**Phase 2:** Traditional Machine Learning Models

## Experiment 2: Time Features

This notebook evaluates the impact of **temporal features** on energy consumption prediction.

**Data:** `netop_ml_time.csv` (22 columns)
- **Raw features (13):** load, ESMode1-6, RUType, Mode, Frequency, Bandwidth, Antennas, TXpower
- **Time features (5):** hour_of_day, day_of_week, is_weekend, is_peak_hour, is_night_time

**Goal:** Measure the value of time-based patterns in energy prediction

## Models Evaluated
- Linear Regression (sanity check)
- Random Forest
- XGBoost
- LightGBM

## Expected Outcome
**10-25% improvement in MAE** over Baseline (Experiment 1)

Time features should capture daily/weekly patterns in network usage and energy consumption.

---
## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Gradient Boosting
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

---
## 2. Helper Functions

In [ ]:
def calculate_mape(y_true, y_pred):
    """Calculate Mean Absolute Percentage Error"""
    # Avoid division by zero
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate_model(y_true, y_pred, dataset_name=""):
    """Calculate all evaluation metrics"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = calculate_mape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    results = {
        'Dataset': dataset_name,
        'MAE (W)': mae,
        'RMSE (W)': rmse,
        'MAPE (%)': mape,
        'R²': r2
    }
    
    return results

def plot_predictions(y_true, y_pred, title="Actual vs Predicted", sample_size=1000):
    """Plot actual vs predicted values"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Scatter plot (sample for visibility)
    idx = np.random.choice(len(y_true), min(sample_size, len(y_true)), replace=False)
    axes[0].scatter(y_true[idx], y_pred[idx], alpha=0.3, s=10)
    axes[0].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 
                 'r--', linewidth=2, label='Perfect Prediction')
    axes[0].set_xlabel('Actual Energy (W)')
    axes[0].set_ylabel('Predicted Energy (W)')
    axes[0].set_title(f'{title}\nScatter Plot')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Residual plot
    residuals = y_true - y_pred
    axes[1].scatter(y_pred[idx], residuals[idx], alpha=0.3, s=10)
    axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
    axes[1].set_xlabel('Predicted Energy (W)')
    axes[1].set_ylabel('Residuals (W)')
    axes[1].set_title(f'{title}\nResidual Plot')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("✓ Helper functions defined")

---
## 3. Load Data

In [ ]:
print("="*80)
print("LOADING TIME FEATURES DATA")
print("="*80)

# Load time features dataset
df_time = pd.read_csv('../processed_data/netop_ml_time.csv')

print(f"\n✓ Time Features: {df_time.shape[0]:,} rows × {df_time.shape[1]} columns")

# Verify no missing values
print(f"\n🔍 Data Quality Check:")
print(f"  Missing values: {df_time.isnull().sum().sum()}")

# Show column names
print(f"\n📋 Columns ({len(df_time.columns)}):")
print(f"  {df_time.columns.tolist()}")

# Show energy statistics
print(f"\n📊 Target Variable (Energy) Statistics:")
print(f"  Mean:   {df_time['Energy'].mean():.2f} W")
print(f"  Std:    {df_time['Energy'].std():.2f} W")
print(f"  Min:    {df_time['Energy'].min():.2f} W")
print(f"  Max:    {df_time['Energy'].max():.2f} W")
print(f"  Median: {df_time['Energy'].median():.2f} W")

# Show sample
print(f"\n📄 Sample Data (first 3 rows):")
print(df_time.head(3))

---
## 4. Experiment 2: Time Features

**Data:** `netop_ml_time.csv` (22 columns)

**Features:** Raw (13) + Time (5) = 18 features
- Raw: load, ESMode1-6, RUType, Mode, Frequency, Bandwidth, Antennas, TXpower
- Time: hour_of_day, day_of_week, is_weekend, is_peak_hour, is_night_time

**Goal:** Measure improvement from temporal patterns

In [ ]:
print("="*80)
print("EXPERIMENT 2: TIME FEATURES")
print("="*80)

# Separate features and target
exclude_cols = ['Time', 'BS', 'CellName', 'Energy']
feature_cols_time = [col for col in df_time.columns if col not in exclude_cols]

print(f"\n📋 Features ({len(feature_cols_time)}):")
print(f"  {feature_cols_time}")

X_time = df_time[feature_cols_time]
y_time = df_time['Energy']

print(f"\n✓ X shape: {X_time.shape}")
print(f"✓ y shape: {y_time.shape}")

### 4.1 Preprocessing Pipeline Setup

In [ ]:
# Identify numerical and categorical features
categorical_features = ['RUType', 'Mode']
numerical_features = [col for col in feature_cols_time if col not in categorical_features]

print(f"\n🔢 Numerical features ({len(numerical_features)}): {numerical_features}")
print(f"🏷️  Categorical features ({len(categorical_features)}): {categorical_features}")

# Create preprocessing pipeline
preprocessor_time = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse=False), categorical_features)
    ])

print("\n✓ Preprocessing pipeline created")

### 4.2 Time Series Cross-Validation Setup

In [ ]:
# TimeSeriesSplit with 5 folds
tscv = TimeSeriesSplit(n_splits=5)

print("\n📅 Time Series Cross-Validation Setup:")
print(f"  Number of splits: 5")
print(f"  Total samples: {len(X_time):,}")
print(f"\n  Fold sizes:")

for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_time), 1):
    print(f"    Fold {fold_idx}: Train={len(train_idx):>6,} | Val={len(val_idx):>6,}")

### 4.3 Model 1: Linear Regression

In [ ]:
print("\n" + "="*60)
print("Model 1: Linear Regression (Time Features)")
print("="*60)

# Create pipeline
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor_time),
    ('regressor', LinearRegression())
])

# Cross-validation
lr_results = []
for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_time), 1):
    # Split data
    X_train, X_val = X_time.iloc[train_idx], X_time.iloc[val_idx]
    y_train, y_val = y_time.iloc[train_idx], y_time.iloc[val_idx]
    
    # Train
    lr_pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = lr_pipeline.predict(X_val)
    
    # Evaluate
    results = evaluate_model(y_val.values, y_pred, f"Fold {fold_idx}")
    lr_results.append(results)
    
    print(f"  Fold {fold_idx}: MAE={results['MAE (W)']:>6.2f} W | RMSE={results['RMSE (W)']:>6.2f} W | R²={results['R²']:>6.4f}")

# Average results
lr_avg = pd.DataFrame(lr_results).mean()
print(f"\n  {'='*58}")
print(f"  Average: MAE={lr_avg['MAE (W)']:>6.2f} W | RMSE={lr_avg['RMSE (W)']:>6.2f} W | R²={lr_avg['R²']:>6.4f}")
print(f"  {'='*58}")

# Train on full dataset for final evaluation
lr_pipeline.fit(X_time, y_time)
y_pred_lr = lr_pipeline.predict(X_time)

print("\n✓ Linear Regression training complete")

### 4.4 Model 2: Random Forest

In [ ]:
print("\n" + "="*60)
print("Model 2: Random Forest (Time Features)")
print("="*60)

# Create pipeline
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor_time),
    ('regressor', RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_split=10,
        min_samples_leaf=4,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0
    ))
])

# Cross-validation
rf_results = []
for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_time), 1):
    print(f"  Training Fold {fold_idx}...", end=" ")
    
    # Split data
    X_train, X_val = X_time.iloc[train_idx], X_time.iloc[val_idx]
    y_train, y_val = y_time.iloc[train_idx], y_time.iloc[val_idx]
    
    # Train
    rf_pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = rf_pipeline.predict(X_val)
    
    # Evaluate
    results = evaluate_model(y_val.values, y_pred, f"Fold {fold_idx}")
    rf_results.append(results)
    
    print(f"MAE={results['MAE (W)']:>6.2f} W | RMSE={results['RMSE (W)']:>6.2f} W | R²={results['R²']:>6.4f}")

# Average results
rf_avg = pd.DataFrame(rf_results).mean()
print(f"\n  {'='*58}")
print(f"  Average: MAE={rf_avg['MAE (W)']:>6.2f} W | RMSE={rf_avg['RMSE (W)']:>6.2f} W | R²={rf_avg['R²']:>6.4f}")
print(f"  {'='*58}")

# Train on full dataset for final evaluation
print("\n  Training on full dataset...")
rf_pipeline.fit(X_time, y_time)
y_pred_rf = rf_pipeline.predict(X_time)

print("\n✓ Random Forest training complete")

### 4.5 Model 3: XGBoost

In [ ]:
print("\n" + "="*60)
print("Model 3: XGBoost (Time Features)")
print("="*60)

# Create pipeline
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor_time),
    ('regressor', XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    ))
])

# Cross-validation
xgb_results = []
for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_time), 1):
    print(f"  Training Fold {fold_idx}...", end=" ")
    
    # Split data
    X_train, X_val = X_time.iloc[train_idx], X_time.iloc[val_idx]
    y_train, y_val = y_time.iloc[train_idx], y_time.iloc[val_idx]
    
    # Train
    xgb_pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = xgb_pipeline.predict(X_val)
    
    # Evaluate
    results = evaluate_model(y_val.values, y_pred, f"Fold {fold_idx}")
    xgb_results.append(results)
    
    print(f"MAE={results['MAE (W)']:>6.2f} W | RMSE={results['RMSE (W)']:>6.2f} W | R²={results['R²']:>6.4f}")

# Average results
xgb_avg = pd.DataFrame(xgb_results).mean()
print(f"\n  {'='*58}")
print(f"  Average: MAE={xgb_avg['MAE (W)']:>6.2f} W | RMSE={xgb_avg['RMSE (W)']:>6.2f} W | R²={xgb_avg['R²']:>6.4f}")
print(f"  {'='*58}")

# Train on full dataset for final evaluation
print("\n  Training on full dataset...")
xgb_pipeline.fit(X_time, y_time)
y_pred_xgb = xgb_pipeline.predict(X_time)

print("\n✓ XGBoost training complete")

### 4.6 Model 4: LightGBM

In [ ]:
print("\n" + "="*60)
print("Model 4: LightGBM (Time Features)")
print("="*60)

# Create pipeline
lgbm_pipeline = Pipeline([
    ('preprocessor', preprocessor_time),
    ('regressor', LGBMRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ))
])

# Cross-validation
lgbm_results = []
for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_time), 1):
    print(f"  Training Fold {fold_idx}...", end=" ")
    
    # Split data
    X_train, X_val = X_time.iloc[train_idx], X_time.iloc[val_idx]
    y_train, y_val = y_time.iloc[train_idx], y_time.iloc[val_idx]
    
    # Train
    lgbm_pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = lgbm_pipeline.predict(X_val)
    
    # Evaluate
    results = evaluate_model(y_val.values, y_pred, f"Fold {fold_idx}")
    lgbm_results.append(results)
    
    print(f"MAE={results['MAE (W)']:>6.2f} W | RMSE={results['RMSE (W)']:>6.2f} W | R²={results['R²']:>6.4f}")

# Average results
lgbm_avg = pd.DataFrame(lgbm_results).mean()
print(f"\n  {'='*58}")
print(f"  Average: MAE={lgbm_avg['MAE (W)']:>6.2f} W | RMSE={lgbm_avg['RMSE (W)']:>6.2f} W | R²={lgbm_avg['R²']:>6.4f}")
print(f"  {'='*58}")

# Train on full dataset for final evaluation
print("\n  Training on full dataset...")
lgbm_pipeline.fit(X_time, y_time)
y_pred_lgbm = lgbm_pipeline.predict(X_time)

print("\n✓ LightGBM training complete")

### 4.7 Experiment 2 Summary

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 2: TIME FEATURES - SUMMARY")
print("="*80)

# Compile results
exp2_summary = pd.DataFrame([
    {'Model': 'Linear Regression', **{k: v for k, v in lr_avg.items() if k != 'Dataset'}},
    {'Model': 'Random Forest', **{k: v for k, v in rf_avg.items() if k != 'Dataset'}},
    {'Model': 'XGBoost', **{k: v for k, v in xgb_avg.items() if k != 'Dataset'}},
    {'Model': 'LightGBM', **{k: v for k, v in lgbm_avg.items() if k != 'Dataset'}}
])

# Sort by R² descending
exp2_summary = exp2_summary.sort_values('R²', ascending=False)

print("\n📊 Cross-Validation Results (5-fold average):")
print(exp2_summary.to_string(index=False))

# Find best model
best_model_exp2 = exp2_summary.iloc[0]['Model']
best_r2_exp2 = exp2_summary.iloc[0]['R²']
best_mae_exp2 = exp2_summary.iloc[0]['MAE (W)']

print(f"\n🏆 Best Model: {best_model_exp2}")
print(f"   R² = {best_r2_exp2:.4f} | MAE = {best_mae_exp2:.2f} W")

# Check success criteria
mean_energy = y_time.mean()
mae_threshold = 0.20 * mean_energy  # 20% of mean

print(f"\n✅ Success Criteria Check:")
print(f"   R² > 0.7:          {'✓ PASS' if best_r2_exp2 > 0.7 else '✗ FAIL'} ({best_r2_exp2:.4f})")
print(f"   MAE < 20% of mean: {'✓ PASS' if best_mae_exp2 < mae_threshold else '✗ FAIL'} ({best_mae_exp2:.2f} W < {mae_threshold:.2f} W)")

### 4.8 Comparison with Baseline (Experiment 1)

In [ ]:
print("\n" + "="*80)
print("COMPARISON: BASELINE vs TIME FEATURES")
print("="*80)

# Load baseline results
try:
    baseline_results = pd.read_csv('../results/traditional_ml_baseline_results.csv')
    
    print("\n📊 Performance Comparison (Best Models):")
    print("\nBaseline (Experiment 1):")
    print(baseline_results.sort_values('R²', ascending=False).head(1).to_string(index=False))
    
    print("\nTime Features (Experiment 2):")
    print(exp2_summary.head(1).to_string(index=False))
    
    # Calculate improvement
    baseline_best_mae = baseline_results.sort_values('R²', ascending=False).iloc[0]['MAE (W)']
    baseline_best_r2 = baseline_results.sort_values('R²', ascending=False).iloc[0]['R²']
    
    mae_improvement = ((baseline_best_mae - best_mae_exp2) / baseline_best_mae) * 100
    r2_improvement = ((best_r2_exp2 - baseline_best_r2) / baseline_best_r2) * 100
    
    print(f"\n📈 Improvement from Adding Time Features:")
    print(f"   MAE: {mae_improvement:>6.2f}% reduction ({baseline_best_mae:.2f} W → {best_mae_exp2:.2f} W)")
    print(f"   R²:  {r2_improvement:>6.2f}% increase ({baseline_best_r2:.4f} → {best_r2_exp2:.4f})")
    
    # Check if improvement meets expectation
    print(f"\n✅ Expected Improvement: 10-25% MAE reduction")
    if 10 <= mae_improvement <= 25:
        print(f"   ✓ WITHIN TARGET ({mae_improvement:.2f}%)")
    elif mae_improvement > 25:
        print(f"   ✓ EXCEEDS TARGET ({mae_improvement:.2f}%)")
    else:
        print(f"   ✗ BELOW TARGET ({mae_improvement:.2f}%)")
        
except FileNotFoundError:
    print("\n⚠️  Baseline results not found. Run traditional_ml_baseline.ipynb first.")

### 4.9 Visualize Best Model

In [ ]:
# Plot predictions for best model
if best_model_exp2 == 'XGBoost':
    plot_predictions(y_time.values, y_pred_xgb, f"XGBoost - Time Features (R²={best_r2_exp2:.4f})")
elif best_model_exp2 == 'LightGBM':
    plot_predictions(y_time.values, y_pred_lgbm, f"LightGBM - Time Features (R²={best_r2_exp2:.4f})")
elif best_model_exp2 == 'Random Forest':
    plot_predictions(y_time.values, y_pred_rf, f"Random Forest - Time Features (R²={best_r2_exp2:.4f})")
else:
    plot_predictions(y_time.values, y_pred_lr, f"Linear Regression - Time Features (R²={best_r2_exp2:.4f})")

### 4.10 Feature Importance Analysis

In [ ]:
print("="*80)
print("FEATURE IMPORTANCE ANALYSIS - TIME FEATURES")
print("="*80)

# Get feature names after preprocessing
preprocessor_time.fit(X_time)

# Get numerical feature names (unchanged)
num_feature_names = numerical_features

# Get categorical feature names after one-hot encoding
cat_encoder = preprocessor_time.named_transformers_['cat']
cat_feature_names = []
for i, cat in enumerate(categorical_features):
    categories = cat_encoder.categories_[i][1:]  # Skip first category
    cat_feature_names.extend([f"{cat}_{cat_val}" for cat_val in categories])

# Combine all feature names
all_feature_names = num_feature_names + cat_feature_names

print(f"\n📋 Total features after preprocessing: {len(all_feature_names)}")
print(f"   Numerical: {len(num_feature_names)}")
print(f"   Categorical (one-hot): {len(cat_feature_names)}")

# Identify time features
time_feature_names = ['hour_of_day', 'day_of_week', 'is_weekend', 'is_peak_hour', 'is_night_time']
print(f"\n⏰ Time features: {time_feature_names}")

# Extract feature importance from each model
print("\n" + "="*80)
print("Extracting Feature Importances...")
print("="*80)

# XGBoost
xgb_importance = xgb_pipeline.named_steps['regressor'].feature_importances_
xgb_fi_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': xgb_importance
}).sort_values('Importance', ascending=False)

print("\n🥇 XGBoost - Top 10 Features:")
print(xgb_fi_df.head(10).to_string(index=False))

# LightGBM
lgbm_importance = lgbm_pipeline.named_steps['regressor'].feature_importances_
lgbm_fi_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': lgbm_importance
}).sort_values('Importance', ascending=False)

print("\n🥈 LightGBM - Top 10 Features:")
print(lgbm_fi_df.head(10).to_string(index=False))

# Random Forest
rf_importance = rf_pipeline.named_steps['regressor'].feature_importances_
rf_fi_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': rf_importance
}).sort_values('Importance', ascending=False)

print("\n🥉 Random Forest - Top 10 Features:")
print(rf_fi_df.head(10).to_string(index=False))

# Highlight time features
print("\n" + "="*80)
print("TIME FEATURES RANKING")
print("="*80)

print("\n⏰ Time Features Importance (XGBoost):")
xgb_time_features = xgb_fi_df[xgb_fi_df['Feature'].isin(time_feature_names)]
print(xgb_time_features.to_string(index=False))

print("\n⏰ Time Features Importance (LightGBM):")
lgbm_time_features = lgbm_fi_df[lgbm_fi_df['Feature'].isin(time_feature_names)]
print(lgbm_time_features.to_string(index=False))

print("\n⏰ Time Features Importance (Random Forest):")
rf_time_features = rf_fi_df[rf_fi_df['Feature'].isin(time_feature_names)]
print(rf_time_features.to_string(index=False))

In [ ]:
# Visualize feature importance comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# XGBoost
top_n = 15
xgb_top = xgb_fi_df.head(top_n)
colors = ['#f39c12' if feat in time_feature_names else '#2ecc71' for feat in xgb_top['Feature']]
axes[0].barh(range(len(xgb_top)), xgb_top['Importance'], color=colors)
axes[0].set_yticks(range(len(xgb_top)))
axes[0].set_yticklabels(xgb_top['Feature'])
axes[0].invert_yaxis()
axes[0].set_xlabel('Importance')
axes[0].set_title(f'XGBoost - Top {top_n} Features\n(R² = {best_r2_exp2:.4f})', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# LightGBM
lgbm_top = lgbm_fi_df.head(top_n)
colors = ['#f39c12' if feat in time_feature_names else '#3498db' for feat in lgbm_top['Feature']]
axes[1].barh(range(len(lgbm_top)), lgbm_top['Importance'], color=colors)
axes[1].set_yticks(range(len(lgbm_top)))
axes[1].set_yticklabels(lgbm_top['Feature'])
axes[1].invert_yaxis()
axes[1].set_xlabel('Importance')
lgbm_r2 = exp2_summary[exp2_summary['Model'] == 'LightGBM']['R²'].values[0]
axes[1].set_title(f'LightGBM - Top {top_n} Features\n(R² = {lgbm_r2:.4f})', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

# Random Forest
rf_top = rf_fi_df.head(top_n)
colors = ['#f39c12' if feat in time_feature_names else '#e74c3c' for feat in rf_top['Feature']]
axes[2].barh(range(len(rf_top)), rf_top['Importance'], color=colors)
axes[2].set_yticks(range(len(rf_top)))
axes[2].set_yticklabels(rf_top['Feature'])
axes[2].invert_yaxis()
axes[2].set_xlabel('Importance')
rf_r2 = exp2_summary[exp2_summary['Model'] == 'Random Forest']['R²'].values[0]
axes[2].set_title(f'Random Forest - Top {top_n} Features\n(R² = {rf_r2:.4f})', fontsize=12, fontweight='bold')
axes[2].grid(axis='x', alpha=0.3)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#f39c12', label='Time Features'),
    Patch(facecolor='#95a5a6', label='Raw Features')
]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.98))

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print("\n✓ Feature importance visualization complete")

In [ ]:
# Consensus analysis
print("\n" + "="*80)
print("CONSENSUS FEATURE IMPORTANCE")
print("="*80)

# Normalize importances
xgb_fi_df['Importance_Norm'] = xgb_fi_df['Importance'] / xgb_fi_df['Importance'].max()
lgbm_fi_df['Importance_Norm'] = lgbm_fi_df['Importance'] / lgbm_fi_df['Importance'].max()
rf_fi_df['Importance_Norm'] = rf_fi_df['Importance'] / rf_fi_df['Importance'].max()

# Create consensus dataframe
consensus_df = pd.DataFrame({
    'Feature': all_feature_names,
    'XGBoost': xgb_fi_df.set_index('Feature')['Importance_Norm'],
    'LightGBM': lgbm_fi_df.set_index('Feature')['Importance_Norm'],
    'RandomForest': rf_fi_df.set_index('Feature')['Importance_Norm']
})

# Calculate average
consensus_df['Average'] = consensus_df[['XGBoost', 'LightGBM', 'RandomForest']].mean(axis=1)
consensus_df['Std'] = consensus_df[['XGBoost', 'LightGBM', 'RandomForest']].std(axis=1)
consensus_df = consensus_df.sort_values('Average', ascending=False)

print("\n📊 Top 15 Features by Average Importance:")
print(consensus_df[['Feature', 'Average', 'Std', 'XGBoost', 'LightGBM', 'RandomForest']].head(15).to_string(index=False))

# Visualize consensus
fig, ax = plt.subplots(figsize=(12, 8))
top_consensus = consensus_df.head(15)
x = range(len(top_consensus))
colors = ['#f39c12' if feat in time_feature_names else '#9b59b6' for feat in top_consensus['Feature']]

ax.barh(x, top_consensus['Average'], xerr=top_consensus['Std'], 
        color=colors, alpha=0.7, capsize=5)
ax.set_yticks(x)
ax.set_yticklabels(top_consensus['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Normalized Importance (Average ± Std)')
ax.set_title('Consensus Feature Importance\nAcross XGBoost, LightGBM, and Random Forest', 
             fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add legend
legend_elements = [
    Patch(facecolor='#f39c12', alpha=0.7, label='Time Features'),
    Patch(facecolor='#9b59b6', alpha=0.7, label='Raw Features')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

print("\n✓ Consensus analysis complete")

---
## 5. Save Experiment 2 Results

In [ ]:
# Save results to CSV
exp2_summary.to_csv('../results/traditional_ml_time_results.csv', index=False)
print("✓ Experiment 2 results saved to ../results/traditional_ml_time_results.csv")

# Save best model
import joblib
if best_model_exp2 == 'XGBoost':
    joblib.dump(xgb_pipeline, '../models/xgboost_time.pkl')
    print("✓ Best model (XGBoost) saved to ../models/xgboost_time.pkl")
elif best_model_exp2 == 'LightGBM':
    joblib.dump(lgbm_pipeline, '../models/lightgbm_time.pkl')
    print("✓ Best model (LightGBM) saved to ../models/lightgbm_time.pkl")
elif best_model_exp2 == 'Random Forest':
    joblib.dump(rf_pipeline, '../models/rf_time.pkl')
    print("✓ Best model (Random Forest) saved to ../models/rf_time.pkl")
else:
    joblib.dump(lr_pipeline, '../models/lr_time.pkl')
    print("✓ Best model (Linear Regression) saved to ../models/lr_time.pkl")

# Save feature importance
consensus_df.to_csv('../results/time_features_importance.csv', index=False)
print("✓ Feature importance saved to ../results/time_features_importance.csv")

---
## Summary & Next Steps

### Key Findings
1. **Time features added value** - Measured improvement in MAE and R² over baseline
2. **Best time features** - Identified which temporal patterns are most predictive
3. **Model performance** - All tree-based models benefited from time features

### Next: Experiment 3 (Full Features)

Continue with `netop_ml_full.csv` to measure the value of lag and rolling window features.

**Expected:** Additional 5-15% improvement in MAE over Time Features